# Spark 4 Cluster - Notebook de verificacion

Comprueba la conexion con todos los servicios: Spark 4, Hive/MySQL, MinIO (S3)
Incluye ejemplos de novedades de Spark 4: VARIANT, SQL UDFs, pipe syntax.

## Crear SparkSession con Hive + S3

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("spark://iabd-spark-master:7077")
    .appName("spark4-full-stack-demo")
    # --- Hive Metastore en MySQL (config explicita) ---
    .config("spark.sql.catalogImplementation", "hive")
    .config("spark.sql.warehouse.dir", "s3a://warehouse/hive")
    .config("spark.hadoop.javax.jdo.option.ConnectionURL",
            "jdbc:mysql://iabd-mysql:3306/hive_metastore?useSSL=false&allowPublicKeyRetrieval=true")
    .config("spark.hadoop.javax.jdo.option.ConnectionDriverName",
            "com.mysql.cj.jdbc.Driver")
    .config("spark.hadoop.javax.jdo.option.ConnectionUserName", "hive")
    .config("spark.hadoop.javax.jdo.option.ConnectionPassword", "hivepass")
    .config("spark.hadoop.hive.metastore.schema.verification", "false")
    .config("spark.hadoop.hive.metastore.schema.verification.record.version", "false")
    .config("spark.hadoop.datanucleus.schema.autoCreateAll", "false")
    .config("spark.hadoop.datanucleus.autoCreateSchema", "false")
    .config("spark.hadoop.datanucleus.fixedDatastore", "true")
    .config("spark.hadoop.datanucleus.schema.autoCreateTables", "false")
    .config("spark.hadoop.datanucleus.schema.validateTables", "false")
    .config("spark.hadoop.datanucleus.schema.validateConstraints", "false")
    .config("spark.hadoop.datanucleus.schema.validateColumns", "false")
    # --- JDBC driver + extra jars (driver y executors) ---
    .config("spark.driver.extraClassPath", "/opt/spark/extra-jars/*:/opt/spark/jars/*")
    .config("spark.executor.extraClassPath", "/opt/spark/extra-jars/*:/opt/spark/jars/*")
    .config("spark.executor.memory", "4g")
    .config("spark.driver.memory", "4g")
    .config("spark.network.timeout", "300s")
    .config("spark.executor.heartbeatInterval", "60s")
        
    # --- MinIO / S3A ---
    .config("spark.hadoop.fs.s3a.endpoint", "http://iabd-minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
        
    # --- Paquetes (Kafka) ---
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.0")
        
    # --- ANSI mode (por defecto en Spark 4) ---
    .config("spark.sql.ansi.enabled", "true")
    .enableHiveSupport()
    .getOrCreate()
)

print(f"Spark version: {spark.version}")
print(f"Spark UI (driver): http://localhost:4040")
print(f"Spark Master UI:   http://localhost:8080")

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/spark/.ivy2.5.2/cache
The jars for the packages stored in: /home/spark/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-67ddf32d-b32f-47dc-923e-8257afe40b98;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.1.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.1.0 in central
	found org.apache.kafka#kafka-clients;3.9.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.4.2 in central
	found org.apache.hadoop#hadoop-client-api;3.4.2 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.modules#scala-parallel-collection

Spark version: 4.1.1
Spark UI (driver): http://localhost:4040
Spark Master UI:   http://localhost:8080


## 4. Kafka - Producir y consumir mensajes

In [2]:
spark.sql("USE demo")
print("USE demo OK")

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


USE demo OK


In [3]:
# %%
from pyspark.sql.functions import to_json, struct, col

df_kafka = (
    spark.sql("SELECT * FROM demo.ventas")
    .select(
        col("id").cast("string").alias("key"),
        to_json(struct("*")).alias("value")
    )
)

(
    df_kafka.write
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("topic", "ventas-eventos")
    .save()
)

print("Mensajes enviados a Kafka topic 'ventas-eventos'")

# %%
# Leer desde Kafka (batch)
df_from_kafka = (
    spark.read
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "ventas-eventos")
    .option("startingOffsets", "earliest")
    .load()
)

df_from_kafka.selectExpr(
    "CAST(key AS STRING)",
    "CAST(value AS STRING)",
    "topic",
    "partition",
    "offset",
    "timestamp"
).show(truncate=False)

26/04/12 17:50:20 WARN TaskSetManager: Lost task 0.0 in stage 0.0 (TID 0) (172.29.0.9 executor 0): org.apache.kafka.common.KafkaException: Failed to construct kafka producer
	at org.apache.kafka.clients.producer.KafkaProducer.<init>(KafkaProducer.java:476)
	at org.apache.kafka.clients.producer.KafkaProducer.<init>(KafkaProducer.java:297)
	at org.apache.kafka.clients.producer.KafkaProducer.<init>(KafkaProducer.java:280)
	at org.apache.spark.sql.kafka010.producer.InternalKafkaProducerPool.createKafkaProducer(InternalKafkaProducerPool.scala:138)
	at org.apache.spark.sql.kafka010.producer.InternalKafkaProducerPool.$anonfun$acquire$1(InternalKafkaProducerPool.scala:84)
	at scala.collection.mutable.HashMap.getOrElseUpdate(HashMap.scala:469)
	at org.apache.spark.sql.kafka010.producer.InternalKafkaProducerPool.acquire(InternalKafkaProducerPool.scala:83)
	at org.apache.spark.sql.kafka010.producer.InternalKafkaProducerPool$.acquire(InternalKafkaProducerPool.scala:200)
	at org.apache.spark.sql.ka

Py4JJavaError: An error occurred while calling o118.save.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 3 in stage 0.0 failed 4 times, most recent failure: Lost task 3.3 in stage 0.0 (TID 12) (172.29.0.9 executor 0): org.apache.kafka.common.KafkaException: Failed to construct kafka producer
	at org.apache.kafka.clients.producer.KafkaProducer.<init>(KafkaProducer.java:476)
	at org.apache.kafka.clients.producer.KafkaProducer.<init>(KafkaProducer.java:297)
	at org.apache.kafka.clients.producer.KafkaProducer.<init>(KafkaProducer.java:280)
	at org.apache.spark.sql.kafka010.producer.InternalKafkaProducerPool.createKafkaProducer(InternalKafkaProducerPool.scala:138)
	at org.apache.spark.sql.kafka010.producer.InternalKafkaProducerPool.$anonfun$acquire$1(InternalKafkaProducerPool.scala:84)
	at scala.collection.mutable.HashMap.getOrElseUpdate(HashMap.scala:469)
	at org.apache.spark.sql.kafka010.producer.InternalKafkaProducerPool.acquire(InternalKafkaProducerPool.scala:

In [ ]:
# %% [markdown]
# ## 6. Structured Streaming desde Kafka (opcional)
# Descomenta para probar streaming en tiempo real.

# %%
# streaming_df = (
#     spark.readStream
#     .format("kafka")
#     .option("kafka.bootstrap.servers", "kafka:9092")
#     .option("subscribe", "ventas-eventos")
#     .option("startingOffsets", "latest")
#     .load()
#     .selectExpr("CAST(value AS STRING) as json_value")
# )
#
# query = (
#     streaming_df.writeStream
#     .format("console")
#     .outputMode("append")
#     .option("truncate", False)
#     .start()
# )
#
# import time
# time.sleep(30)
# query.stop()

# %% [markdown]

In [ ]:
# ## 7. Resumen del entorno

# %%
print("=" * 60)
print(" RESUMEN DEL ENTORNO - SPARK 4")
print("=" * 60)
print(f" Spark Master UI:   http://localhost:8080")
print(f" Spark Driver UI:   http://localhost:4040")
print(f" Jupyter Lab:       http://localhost:8888 (token: spark)")
print(f" MinIO Console:     http://localhost:9001 (minioadmin / minioadmin123)")
print(f" Kafka UI:          http://localhost:8081")
print(f" MySQL:             localhost:3306 (hive / hivepass)")
print(f"")
print(f" Spark version:     {spark.version}")
print(f" ANSI mode:         {spark.conf.get('spark.sql.ansi.enabled')}")
print(f" Workers activos:   {spark.sparkContext.defaultParallelism}")
print("=" * 60)